In [1]:
#@title Setup (helpers)
import json
from typing import Dict, Optional, Tuple, List

import requests
import pandas as pd
from IPython.display import display

# --------------------
# Endpoints
# --------------------
UNIPROT_ENTRY_URL     = "https://rest.uniprot.org/uniprotkb/{acc}.json"
AFDB_PREDICTION_URL   = "https://alphafold.ebi.ac.uk/api/prediction/{af_id}"
PROTVAR_MAP_BY_ACC    = "https://www.ebi.ac.uk/ProtVar/api/mapping/accession/{acc}?page={page}&pageSize={page_size}"
PROTVAR_API_BASE = "https://www.ebi.ac.uk/ProtVar/api"

# --------------------
# HTTP
# --------------------
def http_json(url: str, *, timeout: int = 30) -> dict:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()

def http_text(url: str, *, timeout: int = 60) -> str:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text

# --------------------
# AFDB ID & metadata
# --------------------
def build_afdb_id(uniprot_acc: str, isoform: Optional[str]) -> str:
    iso = (isoform or "").strip()
    return f"AF-{uniprot_acc}-{iso}-F1" if iso else f"AF-{uniprot_acc}-F1"

def afdb_prediction_meta(uniprot_acc: str, isoform: Optional[str]) -> dict:
    af_id = build_afdb_id(uniprot_acc, isoform)
    js = http_json(AFDB_PREDICTION_URL.format(af_id=af_id))
    if isinstance(js, list):
        js = js[0] if js else {}
    return js

# --------------------
# UniProt sequence + CRC64
# --------------------
def uniprot_seq_and_crc64(uniprot_acc: str) -> Tuple[str, str]:
    entry = http_json(UNIPROT_ENTRY_URL.format(acc=uniprot_acc))
    seq = entry.get("sequence", {}).get("value") or ""
    crc = entry.get("sequence", {}).get("crc64") or ""
    if not seq or not crc:
        raise RuntimeError("UniProt REST entry missing sequence and/or crc64.")
    return seq, crc

# --------------------
# Small helpers used by ProtVar parsing
# --------------------
def _safe_get(d: dict, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k, {})
    return cur if cur != {} else default

def _first(seq, default=None):
    return seq[0] if isinstance(seq, list) and len(seq) else default

def _one_letter(aa: Optional[str]) -> Optional[str]:
    if not aa:
        return None
    s = str(aa).strip()
    return s[:1].upper()

def _build_hgvsp(ref_aa: Optional[str], pos: Optional[int], alt_aa: Optional[str]) -> Optional[str]:
    if ref_aa and pos and alt_aa:
        return f"p.{ref_aa}{pos}{alt_aa}"
    return None

def _json_str(x):
    """Stable JSON for lists/dicts; plain scalars passthrough."""
    if isinstance(x, (list, dict)):
        try:
            return json.dumps(x, ensure_ascii=False, separators=(",", ":"))
        except Exception:
            return str(x)
    return x

def _merge_unique_str(items):
    """Semicolon-join list of strings with de-duplication."""
    seen, out = set(), []
    for v in items or []:
        if v and v not in seen:
            seen.add(v); out.append(v)
    return ";".join(out) if out else None

# --------------------
# ProtVar: fetch + flatten EVERYTHING (CSV-safe)
# --------------------
def protvar_fetch_all_for_accession(
    uniprot_acc: str,
    page_size: int = 1000,
    max_pages: Optional[int] = None,
) -> List[dict]:
    """
    Walk content → inputs[*] → mappings[*] → genes[*] → isoforms[*].
    Return one row per isoform-level variant with all useful child fields promoted to columns.
    Complex children are kept as compact JSON strings for CSV.
    """
    rows: List[dict] = []
    page = 1

    while True:
        url = PROTVAR_MAP_BY_ACC.format(acc=uniprot_acc, page=page, page_size=page_size)
        js = http_json(url)

        content = js.get("content") or {}
        inputs = content.get("inputs") or []

        for inp_idx, inp in enumerate(inputs):
            # Input-level context
            inp_inputStr = inp.get("inputStr")
            inp_type     = inp.get("type")
            inp_format   = inp.get("format")
            inp_messages = _json_str(inp.get("messages"))
            inp_chr      = inp.get("chr")
            inp_pos      = inp.get("pos")
            inp_ref      = inp.get("ref")
            inp_alt      = inp.get("alt")

            for map_idx, mapping in enumerate(inp.get("mappings") or []):
                mapping_raw = _json_str(mapping)

                for gene_idx, gene in enumerate(mapping.get("genes") or []):
                    gene_name   = gene.get("geneName")
                    ensg        = gene.get("ensg")
                    reverse     = gene.get("reverseStrand")
                    refAllele_g = gene.get("refAllele")
                    altAllele_g = gene.get("altAllele")
                    cadd_gene   = gene.get("caddScore")
                    afreq_gene  = gene.get("alleleFreq")
                    rsid_gene   = gene.get("rsId") or gene.get("dbSnpId")

                    for iso_idx, iso in enumerate(gene.get("isoforms") or []):
                        acc2      = iso.get("accession")
                        if acc2 and (acc2 != uniprot_acc):
                            continue

                        canonical = iso.get("canonical")
                        canon_acc = iso.get("canonicalAccession")
                        iso_pos   = iso.get("isoformPosition")
                        refAA     = iso.get("refAA")
                        varAA     = iso.get("variantAA")
                        consq     = iso.get("consequences") or iso.get("consequence")
                        protName  = iso.get("proteinName")
                        refCodon  = iso.get("refCodon")
                        varCodon  = iso.get("variantCodon")
                        cdsPos    = iso.get("cdsPosition")
                        codonCh   = iso.get("codonChange")
                        aaChange  = iso.get("aminoAcidChange")

                        # URIs
                        pop_uri   = iso.get("populationObservationsUri")
                        func_uri  = iso.get("referenceFunctionUri")
                        struc_uri = iso.get("proteinStructureUri")

                        # Predictors
                        am_block  = iso.get("amScore") or {}
                        esm_block = iso.get("esmScore") or {}
                        cons_block= iso.get("conservScore") or {}

                        am_name   = am_block.get("name")
                        am_path   = am_block.get("amPathogenicity")
                        am_class  = am_block.get("amClass")

                        esm_name  = esm_block.get("name")
                        esm_score = esm_block.get("score")

                        cons_name = cons_block.get("name")
                        cons_score= cons_block.get("score")

                        # Translated sequences
                        ensp_all, enst_all = [], []
                        for ts in iso.get("translatedSequences") or []:
                            ensp = ts.get("ensp")
                            if ensp:
                                ensp_all.append(ensp)
                            for tr in ts.get("transcripts") or []:
                                enst = tr.get("enst")
                                if enst:
                                    enst_all.append(enst)
                        ensp_join = _merge_unique_str(ensp_all)
                        enst_join = _merge_unique_str(enst_all)
                        ts_raw    = _json_str(iso.get("translatedSequences"))

                        # Derived
                        try:
                            pos_int = int(iso_pos) if iso_pos is not None else None
                        except Exception:
                            pos_int = None
                        hgvsp = _build_hgvsp(_one_letter(refAA), pos_int, _one_letter(varAA))

                        rows.append({
                            # Identity / derived
                            "uniprot_acc": uniprot_acc,
                            "hgvsp": hgvsp,
                            "isoform_pos": pos_int,
                            "refAA_one": _one_letter(refAA),
                            "altAA_one": _one_letter(varAA),

                            # Input-level
                            "input_inputStr": inp_inputStr,
                            "input_type": inp_type,
                            "input_format": inp_format,
                            "input_messages_json": inp_messages,
                            "input_chr": inp_chr,
                            "input_pos": inp_pos,
                            "input_ref": inp_ref,
                            "input_alt": inp_alt,
                            "input_index": inp_idx,

                            # Mapping-level
                            "mapping_index": map_idx,
                            "mapping_raw_json": mapping_raw,

                            # Gene-level
                            "gene": gene_name,
                            "ensg": ensg,
                            "gene_reverseStrand": reverse,
                            "gene_refAllele": refAllele_g,
                            "gene_altAllele": altAllele_g,
                            "gene_caddScore": cadd_gene,
                            "gene_alleleFreq": afreq_gene,
                            "gene_rsid": rsid_gene,
                            "gene_index": gene_idx,

                            # Isoform-level (protein)
                            "iso_accession": acc2,
                            "iso_canonical": canonical,
                            "iso_canonicalAccession": canon_acc,
                            "iso_isoformPosition": iso_pos,
                            "iso_refAA": refAA,
                            "iso_variantAA": varAA,
                            "iso_consequences": consq,
                            "iso_proteinName": protName,
                            "iso_refCodon": refCodon,
                            "iso_variantCodon": varCodon,
                            "iso_cdsPosition": cdsPos,
                            "iso_codonChange": codonCh,
                            "iso_aminoAcidChange": aaChange,

                            # URIs
                            "iso_populationObservationsUri": pop_uri,
                            "iso_referenceFunctionUri": func_uri,
                            "iso_proteinStructureUri": struc_uri,

                            # Predictors (flatten + raw)
                            "am_name": am_name,
                            "am_pathogenicity": am_path,
                            "am_class": am_class,
                            "am_json": _json_str(am_block),

                            "esm_name": esm_name,
                            "esm_score": esm_score,
                            "esm_json": _json_str(esm_block),

                            "conserv_name": cons_name,
                            "conserv_score": cons_score,
                            "conserv_json": _json_str(cons_block),

                            # Translated sequences
                            "ensp_all": ensp_join,
                            "enst_all": enst_join,
                            "translatedSequences_json": ts_raw,

                            # Full isoform block for auditability
                            "iso_raw_json": _json_str(iso),
                        })

        # Pagination
        total_pages = js.get("totalPages")
        if total_pages is None:
            if len(inputs) < page_size:
                break
        else:
            if page >= int(total_pages):
                break

        page += 1
        if max_pages is not None and page > max_pages:
            break

    return rows

def parse_protvar_items_to_df(items: List[dict], uniprot_acc: str) -> pd.DataFrame:
    """
    Build a DataFrame from the flattened items WITHOUT dropping columns.
    Put a small 'core' view first for readability, then append the rest.
    """
    if not items:
        return pd.DataFrame()

    core_cols = [
        "uniprot_acc", "hgvsp", "isoform_pos", "refAA_one", "altAA_one",
        "iso_consequences", "gene", "ensg", "ensp_all", "enst_all",
        "am_pathogenicity", "am_class", "esm_score", "conserv_score",
        "iso_proteinStructureUri", "iso_referenceFunctionUri", "iso_populationObservationsUri",
        "input_chr", "input_pos", "input_ref", "input_alt",
        "gene_refAllele", "gene_altAllele",
    ]
    df = pd.DataFrame(items)
    for c in core_cols:
        if c not in df.columns:
            df[c] = None
    other_cols = [c for c in df.columns if c not in core_cols]
    return df[core_cols + other_cols]

# --------------------
# Minimal mmCIF pLDDT reader (unchanged)
# --------------------
def plddt_from_mmcif(mmcif_text: str, chain_preference=("A",)) -> Tuple[Dict[int,float], int]:
    lines = mmcif_text.splitlines()
    loop_start = None
    for i, line in enumerate(lines):
        if line.strip().lower() == "loop_":
            j = i+1; headers=[]
            while j < len(lines) and lines[j].strip().startswith("_"):
                headers.append(lines[j].strip()); j += 1
            if any(h.lower().startswith("_atom_site.") for h in headers):
                loop_start = i; break
    if loop_start is None:
        return {}, 0
    j = loop_start+1; headers=[]
    while j < len(lines) and lines[j].strip().startswith("_"):
        headers.append(lines[j].strip()); j += 1
    name_to_ix = {h: ix for ix, h in enumerate(headers)}
    def col(name):
        for k,ix in name_to_ix.items():
            if k.lower()==name.lower(): return ix
        return None
    ix_atom  = col("_atom_site.label_atom_id")
    ix_chain = col("_atom_site.label_asym_id")
    ix_seq   = col("_atom_site.auth_seq_id") or col("_atom_site.label_seq_id")
    ix_b     = col("_atom_site.B_iso_or_equiv")
    if None in (ix_atom, ix_chain, ix_seq, ix_b): return {}, 0
    data_end = j
    while data_end < len(lines) and (lines[data_end].startswith("ATOM") or lines[data_end].startswith("HETATM")):
        data_end += 1
    def collect(pref):
        out={}
        for r in range(j, data_end):
            toks = lines[r].split()
            try:
                if toks[ix_atom]!="CA": continue
                if pref and toks[ix_chain] not in pref: continue
                pos = int(float(toks[ix_seq])); out[pos] = float(toks[ix_b])
            except Exception:
                continue
        return out
    p = collect(chain_preference)
    if not p: p = collect(None)
    return p, (max(p) if p else 0)

def _to_abs_url(rel_or_abs: str) -> str:
    s = str(rel_or_abs or "").strip()
    if not s:
        return ""
    if s.startswith("http://") or s.startswith("https://"):
        return s
    if not s.startswith("/"):
        s = "/" + s
    return PROTVAR_API_BASE + s

def protvar_fetch_population(rel_or_abs_uri: str) -> dict:
    """Fetch a populationObservations payload (returns dict; {} on any error)."""
    url = _to_abs_url(rel_or_abs_uri)
    if not url:
        return {}
    try:
        return http_json(url)
    except Exception:
        return {}

def protvar_parse_population(payload: dict) -> dict:
    """
    Parse a populationObservations response into compact CSV-friendly fields.
    This is defensive; unknown structures become None or JSON strings.
    Returns:
      {
        'pop_sources_str': 'gnomAD;ClinVar;dbSNP',
        'pop_rsids': 'rs123;rs456',
        'pop_max_af': 0.0123,
        'pop_records_json': '<raw compact JSON>',
        'pop_clinvar_significance': 'Pathogenic;Likely_benign'  (if present)
      }
    """
    # Keep a raw compact snapshot for auditability
    out = {
        "pop_sources_str": None,
        "pop_rsids": None,
        "pop_max_af": None,
        "pop_records_json": _json_str(payload),
        "pop_clinvar_significance": None,
    }

    # Common shapes to expect (examples; may evolve):
    # payload = { "records": [ { "source": "gnomAD", "id": "...", "alleleFrequency": 1.2e-4, ... }, ... ] }
    records = payload.get("records") or payload.get("items") or []
    if not isinstance(records, list):
        return out

    srcs, rsids, sigs = set(), set(), set()
    max_af = None

    for rec in records:
        src = rec.get("source") or rec.get("database")
        if src:
            srcs.add(str(src))

        # common identifiers
        for key in ("rsId", "rsID", "dbSnpId", "dbSNP", "id"):
            v = rec.get(key)
            if v:
                rsids.add(str(v))
                break

        # allele frequency fields
        for k in ("alleleFrequency", "af", "frequency"):
            v = rec.get(k)
            if isinstance(v, (int, float)):
                max_af = v if (max_af is None or v > max_af) else max_af
                break

        # ClinVar-like significance (if present)
        for k in ("clinicalSignificance", "clinSig", "significance"):
            v = rec.get(k)
            if isinstance(v, str) and v:
                sigs.add(v)
            elif isinstance(v, list) and v:
                for s in v:
                    if s:
                        sigs.add(str(s))

    out["pop_sources_str"] = ";".join(sorted(srcs)) if srcs else None
    out["pop_rsids"] = ";".join(sorted(rsids)) if rsids else None
    out["pop_max_af"] = max_af
    out["pop_clinvar_significance"] = ";".join(sorted(sigs)) if sigs else None
    return out

def protvar_enrich_with_population(items: List[dict]) -> List[dict]:
    """
    For each row (from protvar_fetch_all_for_accession), call populationObservationsUri
    once per unique URI and merge provenance fields into the row.
    """
    # Collect unique URIs
    uris = set()
    for it in items:
        u = it.get("iso_populationObservationsUri")
        if u:
            uris.add(u)

    # Fetch all unique payloads
    cache: Dict[str, dict] = {}
    for u in uris:
        cache[u] = protvar_fetch_population(u)

    # Merge parsed fields back into copies of items
    out = []
    for it in items:
        u = it.get("iso_populationObservationsUri")
        payload = cache.get(u, {}) if u else {}
        parsed = protvar_parse_population(payload)
        merged = {**it, **parsed}
        out.append(merged)
    return out



In [2]:
#@title Enter a UniProt accession and optional isoform, then hit `Runtime` → `Run all`
uniprot_accession = "O15552"   #@param {type:"string"}
isoform = ""                    #@param {type:"string"}  # e.g. "2" for AF-ACC-2-F1
save_outputs = True             #@param {type:"boolean"}

# Optional ProtVar fetch controls
protvar_page_size = 1000        #@param {type:"integer"}  # 10..1000 (ProtVar caps), we use the max by default
protvar_max_pages = None        #@param {type:"raw"}      # e.g. 3 to limit during testing; None = fetch all pages

uniprot_accession = uniprot_accession.strip()
isoform = isoform.strip() or None

print("UniProt:", uniprot_accession)
print("Isoform:", isoform or "(canonical)")
print("ProtVar page_size:", protvar_page_size, "| max_pages:", protvar_max_pages)

UniProt: O15552
Isoform: (canonical)
ProtVar page_size: 1000 | max_pages: None


In [3]:
# 1) UniProtKB REST (sequence + CRC64 from API)
print("▶ Fetching UniProtKB REST entry…")
seq, up_crc64 = uniprot_seq_and_crc64(uniprot_accession)
print(f"   UniProt length={len(seq)}  CRC64={up_crc64}")

# 2) ProtVar API (variants) → tidy table
print("▶ Fetching ProtVar API (mapping/accession)…")
items = protvar_fetch_all_for_accession(
    uniprot_acc=uniprot_accession,
    page_size=int(protvar_page_size),
    max_pages=protvar_max_pages,
)

items = protvar_enrich_with_population(items)

print(f"   ProtVar items fetched: {len(items)}")

df = parse_protvar_items_to_df(items, uniprot_acc=uniprot_accession)
print(f"   Variants parsed: {len(df)}")
display(df.head(10))

# 3) AFDB: build AF-ID, fetch metadata + mmCIF (single model only)
print("▶ Fetching AFDB prediction metadata by AF-ID…")
afdb_id = build_afdb_id(uniprot_accession, isoform)  # canonical if isoform is blank
meta = http_json(AFDB_PREDICTION_URL.format(af_id=afdb_id))
if isinstance(meta, list):
    meta = meta[0] if meta else {}

seq_checksum = meta.get("sequenceChecksum") or meta.get("sequencechecksum")
cif_url = meta.get("cifUrl") or meta.get("mmCifUrl") or meta.get("modelUrl")
print(f"   AFDB ID={afdb_id}")
print(f"   sequenceChecksum={seq_checksum}")
print(f"   mmCIF URL={cif_url}")

if not seq_checksum:
    raise RuntimeError("AFDB metadata missing 'sequenceChecksum' – cannot compare CRC64.")
if not (cif_url and str(cif_url).endswith(".cif")):
    raise RuntimeError("AFDB metadata missing a valid mmCIF URL.")

# Strict compare (UniProt REST CRC64 vs AFDB sequenceChecksum)
print("▶ Strict CRC64 comparison (UniProt REST vs AFDB sequenceChecksum)…")
if up_crc64.upper() != str(seq_checksum).upper():
    raise RuntimeError("Sequence mismatch (CRC64). UniProt sequence does not match AlphaFold model.")
print("   ✓ CRC64 matches.")

# Download mmCIF (and lightly parse pLDDT just to sanity-check the file)
print("▶ Downloading mmCIF…")
mmcif_text = http_text(str(cif_url))
plddt_by_pos, maxpos = plddt_from_mmcif(mmcif_text)
print(f"   mmCIF loaded. pLDDT entries: {len(plddt_by_pos)} | max position: {maxpos}")

# Optionally save raw artefacts next to the notebook
if save_outputs:
    base = f"{afdb_id}"
    df.to_csv(f"{base}_variants.csv", index=False)
    with open(f"{base}_protvar_raw.json", "w") as f: json.dump(items, f, indent=2)
    with open(f"{base}.cif", "w") as f: f.write(mmcif_text)
    print("Saved:", f"{base}_variants.csv", f"{base}_protvar_raw.json", f"{base}.cif")

▶ Fetching UniProtKB REST entry…
   UniProt length=330  CRC64=F4A8AC6AFBDF1E90
▶ Fetching ProtVar API (mapping/accession)…
   ProtVar items fetched: 2970
   Variants parsed: 2970


,uniprot_acc,hgvsp,isoform_pos,refAA_one,altAA_one,iso_consequences,gene,ensg,ensp_all,enst_all,...,esm_json,conserv_name,conserv_json,translatedSequences_json,iso_raw_json,pop_sources_str,pop_rsids,pop_max_af,pop_records_json,pop_clinvar_significance
0,O15552,p.M1L,1,M,L,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-2.144}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
1,O15552,p.M1L,1,M,L,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-2.144}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
2,O15552,p.M1V,1,M,V,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-2.734}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
3,O15552,p.M1L,1,M,L,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.775}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
4,O15552,p.M1T,1,M,T,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.002}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
5,O15552,p.M1A,1,M,A,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.455}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
6,O15552,p.M1I,1,M,I,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.258}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
7,O15552,p.M1I,1,M,I,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.258}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
8,O15552,p.M1I,1,M,I,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.258}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
9,O15552,p.L2M,2,L,M,missense,FFAR2,ENSG00000126262.5,ENSP

▶ Fetching AFDB prediction metadata by AF-ID…
   AFDB ID=AF-O15552-F1
   sequenceChecksum=F4A8AC6AFBDF1E90
   mmCIF URL=https://alphafold.ebi.ac.uk/files/AF-O15552-F1-model_v6.cif
▶ Strict CRC64 comparison (UniProt REST vs AFDB sequenceChecksum)…
   ✓ CRC64 matches.
▶ Downloading mmCIF…
   mmCIF loaded. pLDDT entries: 330 | max position: 330
Saved: AF-O15552-F1_variants.csv AF-O15552-F1_protvar_raw.json AF-O15552-F1.cif


In [4]:
df

,uniprot_acc,hgvsp,isoform_pos,refAA_one,altAA_one,iso_consequences,gene,ensg,ensp_all,enst_all,...,esm_json,conserv_name,conserv_json,translatedSequences_json,iso_raw_json,pop_sources_str,pop_rsids,pop_max_af,pop_records_json,pop_clinvar_significance
0,O15552,p.M1L,1,M,L,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-2.144}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
1,O15552,p.M1L,1,M,L,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-2.144}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
2,O15552,p.M1V,1,M,V,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-2.734}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
3,O15552,p.M1L,1,M,L,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.775}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
4,O15552,p.M1T,1,M,T,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-3.002}",CONSERV,"{""name"":""CONSERV"",""score"":0.863}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2965,O15552,p.G330V,330,G,V,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-0.177}",CONSERV,"{""name"":""CONSERV"",""score"":0.814}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
2966,O15552,p.G330G,330,G,G,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":-1.34}",CONSERV,"{""name"":""CONSERV"",""score"":0.814}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
2967,O15552,p.G330G,330,G,G,synonymous,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,{},CONSERV,"{""name"":""CONSERV"",""score"":0.814}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""proteinColoca...",None
2968,O15552,p.G330A,330,G,A,missense,FFAR2,ENSG00000126262.5,ENSP00000246549.2;ENSP00000473159.1,ENST00000246549.2;ENST00000599180.3,...,"{""name"":""ESM"",""score"":0.207}",CONSERV,"{""name"":""CONSERV"",""score"":0.814}","[{""ensp"":""ENSP00000246549.2"",""transcripts"":[{""...","{""accession"":""O15552"",""canonical"":true,""canoni...",None,None,None,"{""genomicColocatedVariant"":null,""

In [ ]:
#@title Display 3D structure {run: "auto"}
import py3Dmol

# Params (same feel as ColabFold, minus rank_num/pdb)
color = "lDDT" #@param ["chain", "lDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}

# Discrete pLDDT bands (ChimeraX-like)
PLDDT_BANDS = [
    ("<50", None, 50, "#FF7D45"), # orange
    ("50-70", 50, 70, "#FFDB13"), # yellow
    ("70-90", 70, 90, "#66CBF3"), # light blue
    (">90", 90, None, "#0054D7"), # dark blue
]

def apply_lddt_bands(view):
    # baseline cartoon so the model is visible
    view.setStyle({}, {"cartoon": {}})
    # add coloured styles per pLDDT band using bfactor selection
    for _, lo, hi, hexcol in PLDDT_BANDS:
        sel = {"bfactor": [float(lo), float(hi)]}
        view.addStyle(sel, {"cartoon": {"color": hexcol}})


def show_mmcif(mmcif_text: str, color: str = "lDDT",
               show_sidechains: bool = False,
               show_mainchains: bool = False):
    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    view.addModel(mmcif_text, 'cif')

    # Colouring
    if color == "lDDT":
        apply_lddt_bands(view)
    elif color == "rainbow":
        view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
        # Distinct colour per chain label
        # (3Dmol auto-assigns colours; you can refine later if you want a fixed palette)
        chains = set()
        # target only polymer chains via simple residue presence (fast path)
        view.setStyle({'cartoon': {}})  # base style first
        # recolour per chain (use a pass that finds chains by name)
        # We can't query chain IDs from py3Dmol directly, but CIFs use label_asym_id;
        # 3Dmol understands {'chain':'A'} selectors.
        # Try common chain letters A..Z to paint if present:
        for ch in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
            # Apply style; if chain doesn't exist, it's ignored.
            view.setStyle({'chain': ch}, {'cartoon': {}})

    # Extras
    if show_sidechains:
        BB = ['C','O','N']
        view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                      {'stick':{'colorscheme':"WhiteCarbon",'radius':0.3}})
        view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                      {'sphere':{'colorscheme':"WhiteCarbon",'radius':0.3}})
        view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                      {'stick':{'colorscheme':"WhiteCarbon",'radius':0.3}})
    if show_mainchains:
        BB = ['C','O','N','CA']
        view.addStyle({'atom':BB},{'stick':{'colorscheme':"WhiteCarbon",'radius':0.3}})

    view.zoomTo()
    return view

viewer = show_mmcif(mmcif_text, color=color,
                    show_sidechains=show_sidechains,
                    show_mainchains=show_mainchains)
viewer.show()


In [ ]:
df

In [ ]:
#@title pLDDT legend and quick plots {run: "auto"}
import matplotlib.pyplot as plt
import numpy as np

# reuse PLDDT_BANDS from the viewer cell
def plot_plddt_band_legend():
    labels, lows, highs, colors = zip(*PLDDT_BANDS)
    fig, ax = plt.subplots(figsize=(5, 0.6), dpi=200)
    x = 0
    for lab, lo, hi, col in PLDDT_BANDS:
        lo_v = -np.inf if lo is None else lo
        hi_v = 100 if hi is None else hi
        w = (hi_v - (0 if lo_v==-np.inf else lo_v)) / 100.0
        ax.barh([0], [w], left=[x], color=col, edgecolor='none', height=0.8)
        x += w
    ax.set_yticks([])
    ax.set_xlim(0,1)
    ax.set_xlabel("pLDDT bands")
    # ticks at thresholds
    ax.set_xticks([0.5, 0.7, 0.9])
    ax.set_xticklabels(["50","70","90"])
    # text labels
    x = 0
    for lab, lo, hi, col in PLDDT_BANDS:
        lo_v = 0 if (lo is None) else lo/100.0
        hi_v = 1 if (hi is None) else hi/100.0
        ax.text((lo_v+hi_v)/2, 0, lab, va='center', ha='center', fontsize=8, color="black")
    plt.tight_layout()
    return fig

def plot_plddt_hist(plddt_by_pos: dict):
    vals = [float(v) for v in plddt_by_pos.values()]
    if not vals:
        print("No pLDDT values parsed.")
        return
    fig, ax = plt.subplots(figsize=(4, 2.2), dpi=200)
    ax.hist(vals, bins=20)
    ax.set_xlabel("pLDDT")
    ax.set_ylabel("Residue count")
    ax.set_title("pLDDT distribution")
    plt.tight_layout()
    return fig

# Show discrete band legend when using lDDT colouring
try:
    if color == "lDDT":
        plot_plddt_band_legend(); plt.show()
except NameError:
    pass

# Optional distribution
plot_plddt_hist(plddt_by_pos); plt.show()
